## **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

if IS_COLAB:
    !pip install optuna

Repo already exists — pulling latest changes
fatal: unable to access 'https://github.com/Lv1g1/RecSys-Challenge-2025.git/': Could not resolve host: github.com


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

## **Imports**

In [3]:
import numpy as np
import pandas as pd
import gc

from Challenge.paths import load_holdout_split, XGBOOST_DATAFRAMES
from Challenge.utils import split_into_folds, train_all_models

import importlib
import Challenge.features_engineering
importlib.reload(Challenge.features_engineering)
from Challenge.features_engineering import (
    generate_candidates,
    add_models_features,
    calculate_item_item_features_fast,
    add_embedding_features,
    add_aggregate_features_stats,
    add_user_stats
)

Running on local — storage at: /home/luigi/RecSys


## **Load Data**

In [4]:
# Random seed for reproducibility
RANDOM_SEED = 0xc0ffee

In [5]:
URM_inner, URM_outer = load_holdout_split()
folds = split_into_folds(URM_inner, n_folds=10, random_seed=RANDOM_SEED)

Fold 1/10
URM_all: (27095, 6969)
URM_train: (27095, 6969)
URM_validation: (27095, 6969)

URM_all: 2434446
URM_train: 2191001
URM_validation: 243445
------------------------------
Fold 2/10
URM_all: (27095, 6969)
URM_train: (27095, 6969)
URM_validation: (27095, 6969)

URM_all: 2434446
URM_train: 2191001
URM_validation: 243445
------------------------------
Fold 3/10
URM_all: (27095, 6969)
URM_train: (27095, 6969)
URM_validation: (27095, 6969)

URM_all: 2434446
URM_train: 2191001
URM_validation: 243445
------------------------------
Fold 4/10
URM_all: (27095, 6969)
URM_train: (27095, 6969)
URM_validation: (27095, 6969)

URM_all: 2434446
URM_train: 2191001
URM_validation: 243445
------------------------------
Fold 5/10
URM_all: (27095, 6969)
URM_train: (27095, 6969)
URM_validation: (27095, 6969)

URM_all: 2434446
URM_train: 2191001
URM_validation: 243445
------------------------------
Fold 6/10
URM_all: (27095, 6969)
URM_train: (27095, 6969)
URM_validation: (27095, 6969)

URM_all: 2434446

## **Recommeder List**

In [6]:
from Recommenders.KNN.UserKNNCFRecommender import UserKNNCFRecommender
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender
from Recommenders.GraphBased.P3alphaRecommender import P3alphaRecommender
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender
from Recommenders.EASE_R.EASE_R_Recommender import EASE_R_Recommender
from Recommenders.SLIM.SLIMElasticNetRecommender import MultiThreadSLIM_SLIMElasticNetRecommender
from Recommenders.MatrixFactorization.Cython.MatrixFactorization_Cython import MatrixFactorization_WARP_Cython
from Recommenders.Neural.MultVAE_PyTorch_Recommender import MultVAERecommender_PyTorch_OptimizerMask
from implicit.cpu.als import AlternatingLeastSquares

models_mapping = {
    'SLIMElasticNet': MultiThreadSLIM_SLIMElasticNetRecommender,
    'MultVAE': MultVAERecommender_PyTorch_OptimizerMask,
    'IALS': AlternatingLeastSquares,
    # 'EASE_R': EASE_R_Recommender, Conflict with multithreading
    'MatrixFactorization_WARP': MatrixFactorization_WARP_Cython,
    'RP3beta': RP3betaRecommender,
    'P3alpha': P3alphaRecommender,
    'UserKNN_cosine': UserKNNCFRecommender,
    'UserKNN_tversky': UserKNNCFRecommender,
    'ItemKNN_cosine': ItemKNNCFRecommender,
    'ItemKNN_tversky': ItemKNNCFRecommender
}

candidate_cutoffs = {
    # Target of 10 candidates reached at Step 2 (Actual: 10.28).
    "top_10": {'UserKNN_tversky': 0, 'MultVAE': 0, 'SLIMElasticNet': 10, 'EASE_R': 5, 'IALS': 0},
    # Target of 20 candidates reached at Step 7 (Actual: 22.68).
    "top_20": {'UserKNN_tversky': 0, 'MultVAE': 5, 'SLIMElasticNet': 20, 'EASE_R': 10, 'IALS': 5},
    # Target of 30 candidates reached at Step 12 (Actual: 31.10).
    "top_30": {'UserKNN_tversky': 5, 'MultVAE': 10, 'SLIMElasticNet': 25, 'EASE_R': 20, 'IALS': 5},
    # Target of 40 candidates reached at Step 17 (Actual: 46.50).
    "top_40": {'UserKNN_tversky': 10, 'MultVAE': 15, 'SLIMElasticNet': 40, 'EASE_R': 20, 'IALS': 10},
    # Target of 50 candidates reached at Step 20 (Actual: 50.04).
    "top_50": {'UserKNN_tversky': 10, 'MultVAE': 20, 'SLIMElasticNet': 40, 'EASE_R': 25, 'IALS': 15},
    # Target of 80 candidates reached at Step 32 (Actual: 85.56).
    "top_80": {'UserKNN_tversky': 20, 'MultVAE': 40, 'SLIMElasticNet': 70, 'EASE_R': 40, 'IALS': 25}
}

## **Train Models**

In [ ]:
user_ids = np.arange(URM_inner.shape[0])

models = []
# Train EASE before other models to avoid conflicts with multithreading
for fold_index, (URM_train, _) in enumerate(folds):
    m = train_all_models(URM_train, {'EASE_R': EASE_R_Recommender})[0][1]
    models.append([( 'EASE_R', m )])

# Train other models
slow_model_folder = "folds_10"
slow_models = {'SLIMElasticNet': MultiThreadSLIM_SLIMElasticNetRecommender,
               'MultVAE': MultVAERecommender_PyTorch_OptimizerMask,
               'MatrixFactorization_WARP': MatrixFactorization_WARP_Cython}

for fold_index, (URM_train, _) in enumerate(folds):
    print(f"Training fold {fold_index+1}/{len(folds)}")

    for slow_name, class_obj in slow_models.items():
        if os.path.exists(os.path.join(slow_model_folder, slow_name + f"_{fold_index}.zip")):
            print(f"Model {slow_name} for fold {fold_index} already exists. Skipping training.")

            m = class_obj(URM_train)
            m.load_model(slow_model_folder, slow_name + f"_{fold_index}")
            models[fold_index].append((slow_name, m))

            if slow_name in models_mapping:
                del models_mapping[slow_name]
        else:
            print("  WARNING!!")
            print(f"Training slow model {slow_name} for fold {fold_index}.")
            models_mapping[slow_name] = class_obj

    models[fold_index].extend(train_all_models(URM_train, models_mapping))

  Training model: EASE_R
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 7.40 sec
  Training model: EASE_R
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 6.22 sec
  Training model: EASE_R
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 6.23 sec
  Training model: EASE_R
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 8.50 sec
  Training model: EASE_R
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 8.82 sec
  Training model: EASE_R
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 8.57 sec
  Training model: EASE_R
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 7.65 sec
  Training model: EASE_R
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 8.12 sec
  Training model: EASE_R
EASE_R_Recommender: Fitting model... 
E

/home/luigi/.venvs/recsys/lib/python3.13/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/31 [00:00<?, ?it/s]

  Training model: RP3beta
RP3betaRecommender: Similarity column 6969 (100.0%), 4517.49 column/sec. Elapsed time 1.54 sec
  Training model: P3alpha
P3alphaRecommender: Similarity column 6969 (100.0%), 3794.67 column/sec. Elapsed time 1.84 sec
  Training model: UserKNN_cosine
Similarity column 27095 (100.0%), 2067.62 column/sec. Elapsed time 13.10 sec
  Training model: UserKNN_tversky
Similarity column 27095 (100.0%), 2103.65 column/sec. Elapsed time 12.88 sec
  Training model: ItemKNN_cosine
Similarity column 6969 (100.0%), 5788.19 column/sec. Elapsed time 1.20 sec
  Training model: ItemKNN_tversky
Similarity column 6969 (100.0%), 5620.75 column/sec. Elapsed time 1.24 sec
Training fold 2/10
Model SLIMElasticNet for fold 1 already exists. Skipping training.
SLIMElasticNetRecommender: Loading model from file 'folds_10SLIMElasticNet_1'
SLIMElasticNetRecommender: Loading complete
Model MultVAE for fold 1 already exists. Skipping training.
Model MatrixFactorization_WARP for fold 1 already ex

  0%|          | 0/31 [00:00<?, ?it/s]

...

  0%|          | 0/31 [00:00<?, ?it/s]

  Training model: RP3beta
RP3betaRecommender: Similarity column 6969 (100.0%), 4329.97 column/sec. Elapsed time 1.61 sec
  Training model: P3alpha
P3alphaRecommender: Similarity column 6969 (100.0%), 3634.58 column/sec. Elapsed time 1.92 sec
  Training model: UserKNN_cosine
Similarity column 27095 (100.0%), 2069.17 column/sec. Elapsed time 13.09 sec
  Training model: UserKNN_tversky
Similarity column 27095 (100.0%), 2012.44 column/sec. Elapsed time 13.46 sec
  Training model: ItemKNN_cosine
Similarity column 6969 (100.0%), 5378.33 column/sec. Elapsed time 1.30 sec
  Training model: ItemKNN_tversky
Similarity column 6969 (100.0%), 5312.23 column/sec. Elapsed time 1.31 sec
Training fold 10/10
Model SLIMElasticNet for fold 9 already exists. Skipping training.
SLIMElasticNetRecommender: Loading model from file 'folds_10SLIMElasticNet_9'
SLIMElasticNetRecommender: Loading complete
Model MultVAE for fold 9 already exists. Skipping training.
Model MatrixFactorization_WARP for fold 9 already e

  0%|          | 0/31 [00:00<?, ?it/s]

  Training model: RP3beta
RP3betaRecommender: Similarity column 6969 (100.0%), 4272.27 column/sec. Elapsed time 1.63 sec
  Training model: P3alpha
P3alphaRecommender: Similarity column 6969 (100.0%), 3670.54 column/sec. Elapsed time 1.90 sec
  Training model: UserKNN_cosine
Similarity column 27095 (100.0%), 2044.58 column/sec. Elapsed time 13.25 sec
  Training model: UserKNN_tversky
Similarity column 27095 (100.0%), 1992.84 column/sec. Elapsed time 13.60 sec
  Training model: ItemKNN_cosine
Similarity column 6969 (100.0%), 5344.72 column/sec. Elapsed time 1.30 sec
  Training model: ItemKNN_tversky
Similarity column 6969 (100.0%), 5294.54 column/sec. Elapsed time 1.32 sec
Saving model SLIMElasticNet for fold 0.


TypeError: list indices must be integers or slices, not str

In [8]:
models

[[('EASE_R',
   <Recommenders.EASE_R.EASE_R_Recommender.EASE_R_Recommender at 0x7f3202d097f0>),
  ('SLIMElasticNet',
   <Recommenders.SLIM.SLIMElasticNetRecommender.MultiThreadSLIM_SLIMElasticNetRecommender at 0x7f3202d09940>),
  ('MultVAE',
   <Recommenders.Neural.MultVAE_PyTorch_Recommender.MultVAERecommender_PyTorch_OptimizerMask at 0x7f3202d09a90>),
  ('MatrixFactorization_WARP',
   <Recommenders.MatrixFactorization.Cython.MatrixFactorization_Cython.MatrixFactorization_WARP_Cython at 0x7f3202d0a510>),
  ('IALS', <implicit.cpu.als.AlternatingLeastSquares at 0x7f3202d0a660>),
  ('RP3beta',
   <Recommenders.GraphBased.RP3betaRecommender.RP3betaRecommender at 0x7f31fc100d70>),
  ('P3alpha',
   <Recommenders.GraphBased.P3alphaRecommender.P3alphaRecommender at 0x7f31fc101010>),
  ('UserKNN_cosine',
   <Recommenders.KNN.UserKNNCFRecommender.UserKNNCFRecommender at 0x7f31fc100ec0>),
  ('UserKNN_tversky',
   <Recommenders.KNN.UserKNNCFRecommender.UserKNNCFRecommender at 0x7f3202ceb110>),
  

## **Utility**

In [7]:
def sanity_check(df, verbose=True):
    print("--- STARTING SANITY CHECK ---")
    problems_found = False

    # 0. Check DataFrame is sorted by UserID
    if not df['UserID'].is_monotonic_increasing:
        print("\n[CRITICAL] DataFrame is not sorted by UserID in increasing order.")
        problems_found = True

    # 1. Check for Missing Values (NaN)
    null_counts = df.isnull().sum()
    if null_counts.sum() > 0:
        print("\n[CRITICAL] NaN Values Found:")
        print(null_counts[null_counts > 0])
        problems_found = True
    else:
        if verbose: print("[OK] No NaNs found.")

    # 2. Check for Infinite Values (inf / -inf)
    # Common issue when normalizing by zero variance or dividing scores
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    inf_counts = np.isinf(df[numeric_cols]).sum()
    if inf_counts.sum() > 0:
        print("\n[CRITICAL] Infinite Values Found (Division by Zero?):")
        print(inf_counts[inf_counts > 0])
        problems_found = True
    else:
        if verbose: print("[OK] No Infinite values found.")

    # 3. Check for Duplicates (UserID, ItemID)
    # Stacking fails if you have multiple rows for the same User-Item pair
    if df.duplicated(subset=['UserID', 'ItemID']).any():
        n_dupes = df.duplicated(subset=['UserID', 'ItemID']).sum()
        print(f"\n[CRITICAL] Duplicate (UserID, ItemID) pairs found: {n_dupes}")
        problems_found = True
    else:
        if verbose: print("[OK] Keys (UserID, ItemID) are unique.")

    # 4. Check Data Types (IDs must be int)
    # Merges (pd.merge) often convert ints to float if there were missing keys initially
    if df['UserID'].dtype not in [int, np.int16, np.int32, np.int64]:
        print(f"\n[WARNING] UserID is {df['UserID'].dtype}, expected int. (Did a merge fail?)")
        # Auto-fix attempt
        # df['UserID'] = df['UserID'].astype(int) 
    
    if df['ItemID'].dtype not in [int, np.int16, np.int32, np.int64]:
        print(f"\n[WARNING] ItemID is {df['ItemID'].dtype}, expected int.")

    # 5. Check for Constant Columns (Zero Variance)
    # These crash some implementations of Normalization and add no info to XGBoost
    std_devs = df[numeric_cols].std()
    constant_cols = std_devs[std_devs == 0].index.tolist()
    if len(constant_cols) > 0:
        print("\n[WARNING] The following columns have ZERO variance (Constant values):")
        print(constant_cols)
        print("Recommendation: Drop them.")
    
    # 6. Check Logic (Ranks shouldn't be negative)
    rank_cols = [c for c in df.columns if 'Rank' in c and 'Skew' not in c and 'Kurtosis' not in c]
    if rank_cols:
        min_ranks = df[rank_cols].min()
        if (min_ranks < 0).any():
             print("\n[CRITICAL] Negative Ranks found (Logic Error):")
             print(min_ranks[min_ranks < 0])
             problems_found = True

    if problems_found:
        print("\n--- SANITY CHECK FAILED: Fix errors before training ---")
        # raise ValueError("Data Integrity Issues Found") # Uncomment to force stop
    else:
        print("\n--- SANITY CHECK PASSED: Data is clean ---")

    return not problems_found

In [8]:
def optimize_dataframe_types(df: pd.DataFrame) -> pd.DataFrame:
    """
    Downcasts columns to the smallest possible type to save RAM.
    """
    print("Optimizing memory usage...")
    
    # 1. Drop Constant Columns (e.g. top_200 which is always 1)
    nunique = df.nunique()
    cols_to_drop = nunique[nunique == 1].index
    if len(cols_to_drop) > 0:
        print(f"Dropping constant columns: {list(cols_to_drop)}")
        df.drop(columns=cols_to_drop, inplace=True)

    # 2. Downcast Integers
    # IDs: 27k users -> int16 is safe (max 32767)
    # IDs: 7k items -> int16 is safe
    for col in ['UserID', 'ItemID']:
        if col in df.columns:
            df[col] = df[col].astype(np.int16)

    # Ranks: Max 200 -> int16 is safe
    rank_cols = [c for c in df.columns if 'RankPosition' in c]
    for col in rank_cols:
        df[col] = df[col].astype(np.int16)

    # Stats: Profile Len max 27k -> int16 is safe
    if 'User_Profile_Len' in df.columns:
        df['User_Profile_Len'] = df['User_Profile_Len'].astype(np.int16)
        
    # Popularity: Max 27k -> int16 is safe
    if 'Item_Global_Popularity' in df.columns:
        df['Item_Global_Popularity'] = df['Item_Global_Popularity'].astype(np.int16)

    # Clusters: Max 30 -> int8 is safe
    cluster_cols = [c for c in df.columns if 'Cluster' in c and 'Dist' not in c] # Avoid Dist (float)
    for col in cluster_cols:
        # Cluster Interaction (30*20=600) needs int16
        if 'Interaction' in col:
            df[col] = df[col].astype(np.int16)
        else:
            df[col] = df[col].astype(np.int8)

    # Flags: 0/1 -> int8
    flag_cols = [c for c in df.columns if 'top_' in c or 'Budget' in c or 'Label' in c]
    for col in flag_cols:
        df[col] = df[col].astype(np.int8)
        
    # 3. Downcast Floats (Already done in loop, but double check)
    fcols = df.select_dtypes('float').columns
    for col in fcols:
        df[col] = df[col].astype(np.float32)
        
    return df

## **Create Train Dataframe**

In [18]:
def run_oof_pipeline(folds, models, models_cutoff, n_overlap=2) -> pd.DataFrame:
    """
    Main loop to generate OOF data with configurable overlap.
    
    Args:
        folds: List of (URM_train, URM_val) tuples
        models: List of models per fold
        models_cutoff: Dict of cutoffs
        n_overlap (int): Number of folds each user should be processed in.
                         1 = Standard OOF (each user in 1 fold).
                         2 = Double Diagonal (your previous strategy).
                         N_FOLDS = Full saturation (each user in every fold).
    """
    N_FOLDS = len(folds)
    all_dfs = []
    
    # Get total users from first fold
    n_users = folds[0][0].shape[0]
    all_indices = np.arange(n_users)

    # Pre-calculate user remainders to speed up loop
    # This maps every user to their "base fold" (0 to 9)
    user_base_folds = all_indices % N_FOLDS
    
    for fold_idx, (URM_train, URM_val) in enumerate(folds):
        print(f"\n==================================================")
        print(f"PROCESSING FOLD {fold_idx+1}/{len(folds)}")
        print(f"==================================================")

        # ---------------------------------------------------------
        # GENERALIZED USER ASSIGNMENT
        # ---------------------------------------------------------
        # We need to find which 'base folds' are active for this current fold_idx.
        # User u (base b) is processed in Fold F if F is in [b, b+1, ..., b+n_overlap-1]
        # Inverting this: In Fold F, we accept base b if b = (F - offset) % N
        
        # Calculate the list of "base assignments" that target this fold
        active_bases = [(fold_idx - i) % N_FOLDS for i in range(n_overlap)]

        # Select users who belong to these base assignments
        mask = np.isin(user_base_folds, active_bases)
        target_users = all_indices[mask]
        
        print(f"Overlap: {n_overlap} | Active Base Groups: {active_bases}")
        print(f"Target Users for this fold: {len(target_users)} (approx {n_users * n_overlap / N_FOLDS:.0f})")
        
        if len(target_users) == 0:
            continue
        
        models_list = models[fold_idx]

        # Generate Candidates
        print(f"--- Generating Candidates ---")
        df_fold = generate_candidates(
            URM_train[target_users],
            user_ids=target_users,
            models=models_list,
            models_cutoff=models_cutoff
        )
        print(f"Candidates: {len(df_fold)}")

        # Add Labels
        print(f"--- Adding Labels ---")
        gt_values = URM_val[df_fold['UserID'].values, df_fold['ItemID'].values]
        df_fold['Label'] = (np.array(gt_values).squeeze() > 0).astype(np.int8)
        
        positives = df_fold['Label'].sum()
        print(f"Positives found: {positives} (Ratio: {positives/len(df_fold):.4f})")

        # Add Features
        print(f"--- Adding Features ---")
        
        # 1. Model Scores & Ranks
        df_fold = add_models_features(df_fold, URM_train, models_list)
        
        # 2. Item-Item Similarity (SLIM Only)
        # Filter models list to find SLIM
        slim_models = [m for m in models_list if 'SLIM' in m[0] or 'RP3beta' in m[0]]
        if slim_models:
            df_fold = calculate_item_item_features_fast(df_fold, URM_train, slim_models)
            
        # 3. Embedding Features (IALS Only)
        # Filter to find IALS
        ials_model = next((m[1] for m in models_list if 'IALS' in m[0]), None)
        if ials_model:
            df_fold = add_embedding_features(df_fold, ials_model)
            
        # 4. Aggregates (Mean/Std/Min/Max)
        df_fold = add_aggregate_features_stats(df_fold)
        
        # 5. User Stats (Mainstreamness)
        df_fold = add_user_stats(df_fold, URM_train)

        # Optimize Memory
        print(f"--- Optimizing Types ---")
        df_fold = optimize_dataframe_types(df_fold)
        
        print(f"Final Inference Shape: {df_fold.shape}")
        print(f"Memory Usage: {df_fold.memory_usage().sum() / 1e6:.2f} MB")

        # Check
        assert sanity_check(df_fold), f"Sanity check failed for fold {fold_idx}"

        # G. Accumulate
        all_dfs.append(df_fold)
        
        # H. Cleanup to save RAM
        del df_fold, models_list, URM_train, URM_val
        gc.collect()

    # ---------------------------------------------------------
    # 2. CONCATENATE
    # ---------------------------------------------------------
    print("\nConcatenating all folds...")
    full_train_df = pd.concat(all_dfs, ignore_index=True)
    
    # Sort by UserID for faster access during training
    full_train_df.sort_values(by='UserID', inplace=True)

    print(f"Final OOF Dataset Shape: {full_train_df.shape}")
    print(f"Memory Usage: {full_train_df.memory_usage().sum() / 1e6:.2f} MB")
    return full_train_df

In [19]:
df_train = run_oof_pipeline(folds, models, candidate_cutoffs, n_overlap=10)


PROCESSING FOLD 1/10
Overlap: 10 | Active Base Groups: [0, 9, 8, 7, 6, 5, 4, 3, 2, 1]
Target Users for this fold: 27095 (approx 27095)
--- Generating Candidates ---
Computing candidates with EASE_R...
Computing candidates with EASE_R (Max Cutoff: 40)...
Computing candidates with SLIMElasticNet...
Computing candidates with SLIMElasticNet (Max Cutoff: 70)...
Computing candidates with MultVAE...
Computing candidates with MultVAE (Max Cutoff: 40)...
Computing candidates with MatrixFactorization_WARP...
Skipping MatrixFactorization_WARP (Max cutoff is 0)...
Computing candidates with IALS...
Computing candidates with IALS (Max Cutoff: 25)...
Computing candidates with RP3beta...
Skipping RP3beta (Max cutoff is 0)...
Computing candidates with P3alpha...
Skipping P3alpha (Max cutoff is 0)...
Computing candidates with UserKNN_cosine...
Skipping UserKNN_cosine (Max cutoff is 0)...
Computing candidates with UserKNN_tversky...
Computing candidates with UserKNN_tversky (Max Cutoff: 20)...
Computing

100%|██████████| 27095/27095 [00:08<00:00, 3142.36it/s]


Extracting MAX similarity for RP3beta...


100%|██████████| 27095/27095 [00:06<00:00, 4228.90it/s]


Computing distances in batches of 500...


...

Extracting MAX similarity for RP3beta...


100%|██████████| 27095/27095 [00:06<00:00, 4241.92it/s]


Computing distances in batches of 500...


100%|██████████| 4640/4640 [00:00<00:00, 6640.30it/s]


Assigning columns to DataFrame...
--- Optimizing Types ---
Optimizing memory usage...
Dropping constant columns: ['top_80']
Final Inference Shape: (2319668, 53)
Memory Usage: 338.67 MB
--- STARTING SANITY CHECK ---
[OK] No NaNs found.
[OK] No Infinite values found.
[OK] Keys (UserID, ItemID) are unique.

--- SANITY CHECK PASSED: Data is clean ---

Concatenating all folds...
Final OOF Dataset Shape: (23176694, 53)
Memory Usage: 3569.21 MB


In [20]:
df_train.dtypes

UserID                                     int16
ItemID                                     int16
top_10                                      int8
top_20                                      int8
top_30                                      int8
top_40                                      int8
top_50                                      int8
Label                                       int8
EASE_R_Score                             float32
EASE_R_RankPosition                        int16
SLIMElasticNet_Score                     float32
SLIMElasticNet_RankPosition                int16
MultVAE_Score                            float32
MultVAE_RankPosition                       int16
MatrixFactorization_WARP_Score           float32
MatrixFactorization_WARP_RankPosition      int16
IALS_Score                               float32
IALS_RankPosition                          int16
RP3beta_Score                            float32
RP3beta_RankPosition                       int16
P3alpha_Score       

In [21]:
print(f"Final Memory Usage: {df_train.memory_usage().sum() / 1e6:.2f} MB")

Final Memory Usage: 3569.21 MB


In [22]:
df_train

,UserID,ItemID,top_10,top_20,top_30,top_40,top_50,Label,EASE_R_Score,EASE_R_RankPosition,...,User_to_ItemCluster_Dist,Mean_RankPosition,Std_RankPosition,Min_RankPosition,Mean_Score,Std_Score,Max_Score,User_Profile_Len,Item_Global_Popularity,User_Avg_Item_Popularity
0,0,13,0,0,0,0,0,0,0.146634,106,...,2.763314,646,539,59,0.084832,0.081167,0.221003,77,413,1541.376587
13903167,0,5706,0,0,0,1,1,0,0.180238,58,...,2.756729,827,1427,10,0.191343,0.202138,0.577017,72,485,1429.055542
13903168,0,5766,0,0,1,1,1,0,0.305060,16,...,2.756729,351,1050,16,0.273811,0.139576,0.503647,72,2895,1429.055542
13903169,0,5830,0,0,0,0,0,0,0.162575,75,...,2.777495,412,1056,20,0.230950,0.167745,0.525276,72,1731,1429.055542
13903170,0,5956,0,0,0,0,0,0,0.241022,31,...,2.803360,43,47,10,0.289829,0.098583,0.472416,72,1454,1429.055542
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16221257,27094,640,0,0,0,1,1,0,0.275289,76,...,4.523964,84,51,17,0.359248,0.152304,0.757386,221,2377,874.778259
16221258,27094,797,0,0,0,0,0,0,0.247680,96,...,4.458327,214,356,19,0.349893,0.228300,0.704917,221,615,874.778259
16221259,27094,868,0,0,0,0,0,0,0.316601,55,...,4.585932,706,1181,48,0.239673,0.137379,0.441403,221,6416,874.778259
11585660,27094,101,0,0,0,0,0,0,0.396964,33,...,4.503058,67,64,7,0.413890,0.196016,0.665027,221,576,818.176453


### **Save Dataframe**

In [23]:
# Save
save_path = os.path.join(XGBOOST_DATAFRAMES, "training_OOF.parquet")
os.makedirs(os.path.dirname(save_path), exist_ok=True)

df_train.to_parquet(
    path=save_path,
    engine='fastparquet',
    compression='snappy',
    index=False
)

In [24]:
# Free Memory
del df_train, models
gc.collect()

20

## **Validation Dataframe**

In [9]:
models = train_all_models(URM_inner, models_mapping)
m = train_all_models(URM_inner, {'EASE_R': EASE_R_Recommender})[0][1]
models.append([( 'EASE_R', m )])

  Training model: SLIMElasticNet


100%|█████████▉| 6968/6969 [01:31<00:00, 76.28it/s] 


  Training model: MultVAE


/home/luigi/RecSys/Recommenders/Neural/MultVAE_PyTorch_Recommender.py:330: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  user_batch_tensor = torch.sparse_csr_tensor(user_batch_tensor.indptr,


MultVAERecommender_PyTorch: Epoch 1 of 70. Elapsed time 1.10 sec
MultVAERecommender_PyTorch: Epoch 2 of 70. Elapsed time 1.94 sec
MultVAERecommender_PyTorch: Epoch 3 of 70. Elapsed time 2.79 sec
MultVAERecommender_PyTorch: Epoch 4 of 70. Elapsed time 3.63 sec
MultVAERecommender_PyTorch: Epoch 5 of 70. Elapsed time 4.49 sec
MultVAERecommender_PyTorch: Epoch 6 of 70. Elapsed time 5.35 sec
MultVAERecommender_PyTorch: Epoch 7 of 70. Elapsed time 6.21 sec
MultVAERecommender_PyTorch: Epoch 8 of 70. Elapsed time 7.05 sec
MultVAERecommender_PyTorch: Epoch 9 of 70. Elapsed time 7.93 sec
MultVAERecommender_PyTorch: Epoch 10 of 70. Elapsed time 8.79 sec
MultVAERecommender_PyTorch: Epoch 11 of 70. Elapsed time 9.63 sec
MultVAERecommender_PyTorch: Epoch 12 of 70. Elapsed time 10.47 sec
MultVAERecommender_PyTorch: Epoch 13 of 70. Elapsed time 11.33 sec
MultVAERecommender_PyTorch: Epoch 14 of 70. Elapsed time 12.21 sec
MultVAERecommender_PyTorch: Epoch 15 of 70. Elapsed time 13.06 sec
MultVAERecommen

/home/luigi/.venvs/recsys/lib/python3.13/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/31 [00:00<?, ?it/s]

  Training model: MatrixFactorization_WARP
MF_WARP: Processed 27648 (100.0%) in 0.26 sec. MSE loss 8.32E-02. Sample per second: 105295
MF_WARP: Epoch 1 of 1500. Elapsed time 0.16 sec
MF_WARP: Processed 27648 (100.0%) in 0.42 sec. MSE loss 1.49E-01. Sample per second: 65912
MF_WARP: Epoch 2 of 1500. Elapsed time 0.31 sec
MF_WARP: Processed 27648 (100.0%) in 0.57 sec. MSE loss 1.89E-01. Sample per second: 48607
MF_WARP: Epoch 3 of 1500. Elapsed time 0.46 sec
MF_WARP: Processed 27648 (100.0%) in 0.73 sec. MSE loss 2.38E-01. Sample per second: 38104
MF_WARP: Epoch 4 of 1500. Elapsed time 0.62 sec
MF_WARP: Processed 27648 (100.0%) in 0.88 sec. MSE loss 2.91E-01. Sample per second: 31586
MF_WARP: Epoch 5 of 1500. Elapsed time 0.77 sec
MF_WARP: Processed 27648 (100.0%) in 1.03 sec. MSE loss 3.74E-01. Sample per second: 26920
MF_WARP: Epoch 6 of 1500. Elapsed time 0.92 sec
MF_WARP: Processed 27648 (100.0%) in 0.18 sec. MSE loss 5.10E-01. Sample per second: 157606
MF_WARP: Epoch 7 of 1500. Elap

In [12]:
models

[('SLIMElasticNet',
  <Recommenders.SLIM.SLIMElasticNetRecommender.MultiThreadSLIM_SLIMElasticNetRecommender at 0x7f7ebf0d97f0>),
 ('MultVAE',
  <Recommenders.Neural.MultVAE_PyTorch_Recommender.MultVAERecommender_PyTorch_OptimizerMask at 0x7f7ebf0da510>),
 ('IALS', <implicit.cpu.als.AlternatingLeastSquares at 0x7f7fa4200440>),
 ('MatrixFactorization_WARP',
  <Recommenders.MatrixFactorization.Cython.MatrixFactorization_Cython.MatrixFactorization_WARP_Cython at 0x7f7fa42023c0>),
 ('RP3beta',
  <Recommenders.GraphBased.RP3betaRecommender.RP3betaRecommender at 0x7f7fa4202510>),
 ('P3alpha',
  <Recommenders.GraphBased.P3alphaRecommender.P3alphaRecommender at 0x7f7fa42027b0>),
 ('UserKNN_cosine',
  <Recommenders.KNN.UserKNNCFRecommender.UserKNNCFRecommender at 0x7f7fa4202660>),
 ('UserKNN_tversky',
  <Recommenders.KNN.UserKNNCFRecommender.UserKNNCFRecommender at 0x7f7fa4322210>),
 ('ItemKNN_cosine',
  <Recommenders.KNN.ItemKNNCFRecommender.ItemKNNCFRecommender at 0x7f7fa4202900>),
 ('ItemKNN

In [14]:
models[-1] = models[-1][0]

In [15]:
def generate_inference_data(
        URM_train, 
        target_users, 
        models, 
        models_cutoff):
    """
    Generates the dataframe for Validation or Test.
    
    Args:
        URM_train: The URM used to train the models (e.g. URM_inner).
                   Features like 'Similarity to History' are calculated against this.
        target_users: Array of users to predict for.
        models: List of tuples [('SLIM', model), ...] (Trained on URM_train).
        models_cutoff: The cutoff dictionary.
    """
    
    print(f"Generating inference data for {len(target_users)} users...")
    
    # 1. Generate Candidates
    # IMPORTANT: We slice URM_train[target_users] for ALS compatibility, 
    # just like in the OOF pipeline.
    print(f"--- Generating Candidates ---")
    df = generate_candidates(
        URM_train[target_users],
        user_ids=target_users,
        models=models,
        models_cutoff=models_cutoff
    )
    print(f"Candidates generated: {len(df)}")
    
    # 2. Add Model Features (Scores & Ranks)
    print(f"--- Adding Model Features ---")
    df = add_models_features(df, URM_train, models)
    
    # 3. Add Item-Item Similarity Features (SLIM Only)
    slim_models = [m for m in models if 'SLIM' in m[0] or 'RP3beta' in m[0]]
    if slim_models:
        print(f"--- Adding SLIM Features ---")
        df = calculate_item_item_features_fast(df, URM_train, slim_models)
        
    # 4. Add Embedding Features (IALS Only)
    ials_model = next((m[1] for m in models if 'IALS' in m[0]), None)
    if ials_model:
        print(f"--- Adding IALS Embeddings ---")
        df = add_embedding_features(df, ials_model)
        
    # 5. Add Aggregate Features
    print(f"--- Adding Aggregates ---")
    df = add_aggregate_features_stats(df)
    
    # 6. Add User Stats (Mainstreamness)
    print(f"--- Adding User Stats ---")
    df = add_user_stats(df, URM_train)
    
    # 7. Optimize Memory
    print(f"--- Optimizing Types ---")
    df = optimize_dataframe_types(df)
    
    print(f"Final Inference Shape: {df.shape}")
    print(f"Memory Usage: {df.memory_usage().sum() / 1e6:.2f} MB")
    
    return df

In [16]:
target_users = np.arange(URM_inner.shape[0])
df_val = generate_inference_data(URM_inner, target_users, models, candidate_cutoffs)

Generating inference data for 27095 users...
--- Generating Candidates ---
Computing candidates with SLIMElasticNet...
Computing candidates with SLIMElasticNet (Max Cutoff: 70)...
Computing candidates with MultVAE...
Computing candidates with MultVAE (Max Cutoff: 40)...
Computing candidates with IALS...
Computing candidates with IALS (Max Cutoff: 25)...
Computing candidates with MatrixFactorization_WARP...
Skipping MatrixFactorization_WARP (Max cutoff is 0)...
Computing candidates with RP3beta...
Skipping RP3beta (Max cutoff is 0)...
Computing candidates with P3alpha...
Skipping P3alpha (Max cutoff is 0)...
Computing candidates with UserKNN_cosine...
Skipping UserKNN_cosine (Max cutoff is 0)...
Computing candidates with UserKNN_tversky...
Computing candidates with UserKNN_tversky (Max Cutoff: 20)...
Computing candidates with ItemKNN_cosine...
Skipping ItemKNN_cosine (Max cutoff is 0)...
Computing candidates with ItemKNN_tversky...
Skipping ItemKNN_tversky (Max cutoff is 0)...
Computing

100%|██████████| 27095/27095 [00:08<00:00, 3346.71it/s]


Extracting MAX similarity for RP3beta...


100%|██████████| 27095/27095 [00:05<00:00, 4566.62it/s]


--- Adding IALS Embeddings ---
Computing distances in batches of 500...


100%|██████████| 4593/4593 [00:00<00:00, 7242.26it/s]


Assigning columns to DataFrame...
--- Adding Aggregates ---
--- Adding User Stats ---
--- Optimizing Types ---
Optimizing memory usage...
Dropping constant columns: ['top_80']
Final Inference Shape: (2296118, 52)
Memory Usage: 332.94 MB


In [17]:
# Save
save_path = os.path.join(XGBOOST_DATAFRAMES, "validation_OOF.parquet")
os.makedirs(os.path.dirname(save_path), exist_ok=True)

df_val.to_parquet(
    path=save_path,
    engine='fastparquet',
    compression='snappy',
    index=False
)